In [ ]:
%%capture
import nltk
import pandas as pd
from sklearn.metrics import classification_report
from nltk.classify import NaiveBayesClassifier
import spacy
import sys
!pip install scispacy


In [ ]:
Drug_lib_df = pd.read_csv("drugLibTest_NLP Final Project Dataset.csv")
Drug_lib_df.head()

In [ ]:
print("number of drug_lib transcripts: ", len(Drug_lib_df))

In [ ]:
%%capture
import sys
!{sys.executable} -m pip install spacy
!{sys.executable} -m pip install scispacy

In [ ]:
from nltk.corpus import stopwords  # Stopword removal
from nltk.tokenize import word_tokenize  # Tokenization
from nltk.stem import WordNetLemmatizer  # Lemmatization

In [ ]:
#Check  rows with null values
Drug_lib_df.isnull().sum().sort_values(ascending = False)

In [ ]:
#Remove rows with nulls
Drug_lib_df = Drug_lib_df[Drug_lib_df['sideEffectsReview'].notna()]
Drug_lib_df.info()

In [ ]:
#Remove rows with nulls
Drug_lib_df = Drug_lib_df[Drug_lib_df['benefitsReview'].notna()]
Drug_lib_df.info()

In [ ]:
#Remove rows with nulls
Drug_lib_df = Drug_lib_df[Drug_lib_df['commentsReview'].notna()]
Drug_lib_df.info()

## Convert into Lower Case

In [ ]:
#Convert sideEffectsReview  into lowercase
Drug_lib_df['sideEffectsReview'] = Drug_lib_df['sideEffectsReview'].str.lower()
Drug_lib_df.head()

In [ ]:
#Convert benefitsReview into lowercase
Drug_lib_df['commentsReview'] = Drug_lib_df['commentsReview'].astype(str).str.lower()

Drug_lib_df.head()

In [ ]:
#Convert commentsReview into lowercase
Drug_lib_df['benefitsReview'] = Drug_lib_df['benefitsReview'].str.lower()
Drug_lib_df.head()

In [ ]:
#Convert sideEffects into lowercase
Drug_lib_df['sideEffects'] = Drug_lib_df['sideEffects'].str.lower()
Drug_lib_df.head()

In [ ]:
#Convert effectiveness into lowercase
Drug_lib_df['effectiveness'] = Drug_lib_df['effectiveness'].str.lower()
Drug_lib_df.head()

In [ ]:
#Remove stop words

# Download the 'punkt_tab' resource before tokenization
nltk.download('punkt_tab') # Download the necessary data for punkt tokenizer

word_tokenized = nltk.word_tokenize(Drug_lib_df['sideEffectsReview'][0])
nltk.download('stopwords')
stopwords = nltk.corpus.stopwords.words('english')
word_tokenized = [word for word in word_tokenized if word not in stopwords]
print(word_tokenized)

In [ ]:
sideEffectsReview = Drug_lib_df['sideEffectsReview']
sideEffects = Drug_lib_df ['sideEffects']
data_with_labels =list(zip(sideEffectsReview, sideEffects))
print ("data_with_labels[0]: ", data_with_labels[0])

In [ ]:
urlDrugName = Drug_lib_df['urlDrugName']
condition = Drug_lib_df ['condition']
data_with_labels =list(zip(urlDrugName, condition))
print ("data_with_labels[0]: ", data_with_labels[0])

#### Download Vader Lexicon

In [ ]:
nltk.download('vader_lexicon')

 ## Applying VADER Sentiment Analysis on Comments, Benefits, and Side Effects Reviews

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')

# Initialize VADER
sid = SentimentIntensityAnalyzer()

def label_sentiment(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

columns_to_check = ['commentsReview', 'benefitsReview', 'sideEffectsReview']

for col in columns_to_check:
    print(f"\nSentiment analysis for: {col}")
    Drug_lib_df['vader_compound'] = Drug_lib_df[col].astype(str).apply(lambda x: sid.polarity_scores(x)['compound'])
    Drug_lib_df['vader_label'] = Drug_lib_df['vader_compound'].apply(label_sentiment)
    sentiment_counts = Drug_lib_df['vader_label'].value_counts(normalize=True) * 100
    print(sentiment_counts)

In [ ]:
%%capture
!pip install transformers torch

In [21]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TextClassificationPipeline

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)





from transformers import pipeline

# Load the sentiment analysis pipeline with truncation enabled
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    truncation=True,
)

#  Create Text Classification Pipeline
health_sentiment = TextClassificationPipeline(model=model, tokenizer=tokenizer, return_all_scores=False)


reviews = Drug_lib_df['benefitsReview'].astype(str).tolist()
sentiment_results = sentiment_pipeline(reviews)

# Extract labels and scores
sentiment_labels = [result['label'] for result in sentiment_results]
sentiment_scores = [result['score'] for result in sentiment_results]



Drug_lib_df['bert_sentiment_label'] = sentiment_labels
Drug_lib_df['bert_sentiment_score'] = sentiment_scores


In [22]:
# Select a subset of reviews (first 50 for speed, expand later)
sample_reviews = Drug_lib_df['benefitsReview'].dropna().head().tolist()

# Run sentiment prediction
predicted_sentiment = health_sentiment(sample_reviews)

index_values = Drug_lib_df.head(len(predicted_sentiment)).index

# Add star label to DataFrame using index_values
Drug_lib_df.loc[index_values, 'bert_star_label'] = [res['label'] for res in predicted_sentiment]


In [23]:
def star_to_sentiment(label):
    stars = int(label.split()[0])
    if stars <= 2:
        return 'negative'
    elif stars == 3:
        return 'neutral'
    else:
        return 'positive'

Drug_lib_df.loc[index_values, 'bert_star_label'] = [res['label'] for res in predicted_sentiment]




In [24]:
Drug_lib_df[['benefitsReview', 'bert_star_label', 'bert_sentiment_score']].head()

,benefitsReview,bert_star_label,bert_sentiment_score
0,the antibiotic may have destroyed bacteria cau...,3 stars,0.431390
1,lamictal stabilized my serious mood swings. on...,5 stars,0.633112
2,initial benefits were comparable to the brand ...,4 stars,0.320670
3,it controlls my mood swings. it helps me think...,5 stars,0.763892
4,within one week of treatment superficial acne ...,1 star,0.584940


In [25]:
def get_bert_sentiment(text):
    return sentiment_pipeline(text)[0]['label']

# Apply the function to the dataset
Drug_lib_df['benefitsReview_raw'] = Drug_lib_df['benefitsReview'].apply(
    lambda x: get_bert_sentiment(str(x))
)

# Convert rating (1-5 stars) to positive, neutral, negative
def simplify(label):
    stars = int(label[0])  # Extract the first digit
    if stars >= 4:
        return 'positive'
    elif stars == 3:
        return 'neutral'
    else:
        return 'negative'

Drug_lib_df['benefitsReview_sentiment'] = Drug_lib_df['benefitsReview_raw'].apply(
    simplify
)

# Show results
Drug_lib_df[
    ['benefitsReview', 'benefitsReview_raw', 'benefitsReview_sentiment']
].head()

,benefitsReview,benefitsReview_raw,benefitsReview_sentiment
0,the antibiotic may have destroyed bacteria cau...,3 stars,neutral
1,lamictal stabilized my serious mood swings. on...,5 stars,positive
2,initial benefits were comparable to the brand ...,4 stars,positive
3,it controlls my mood swings. it helps me think...,5 stars,positive
4,within one week of treatment superficial acne ...,1 star,negative


In [ ]:
def get_bert_sentiment(text):
    return sentiment_pipeline(text)[0]['label']

# Apply the function to the dataset
Drug_lib_df['commentsReview_raw'] = Drug_lib_df['commentsReview'].apply(
    lambda x: get_bert_sentiment(str(x))
)

# Convert rating (1-5 stars) to positive, neutral, negative
def simplify(label):
    stars = int(label[0])  # Extract the first digit
    if stars >= 4:
        return 'positive'
    elif stars == 3:
        return 'neutral'
    else:
        return 'negative'

Drug_lib_df['commentsReview_sentiment'] = Drug_lib_df['commentsReview_raw'].apply(
    simplify
)

# Show results
Drug_lib_df[
    ['commentsReview', 'commentsReview_raw', 'commentsReview_sentiment']
].head()

## **NER (Named Entity Recognition)**

In [ ]:
%%capture
import sys
#Model trained on BC5CDR corpus for disease and Chemical entities
!{sys.executable} -m pip install http://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_ner_bc5cdr_md-0.5.3.tar.gz

In [ ]:
import spacy
import en_ner_bc5cdr_md

In [ ]:
nlp = spacy.load("en_ner_bc5cdr_md")

In [ ]:
Drug_lib_df = Drug_lib_df[Drug_lib_df['benefitsReview'].notna()]

# 2. Clean the benefitsReview column (convert to lowercase)
Drug_lib_df['benefitsReview'] = Drug_lib_df['benefitsReview'].astype(str).str.lower()

# 3. Define your NER function
def extract_entities(text):
    if isinstance(text, str):
        doc = nlp(text)
        return [(ent.text, ent.label_) for ent in doc.ents]
    else:
        return []

# 4. Apply entity extraction to each review
Drug_lib_df['medical_entities'] = Drug_lib_df['benefitsReview'].apply(extract_entities)

# 5. Display the results
Drug_lib_df[['benefitsReview', 'medical_entities']].head()


## **Drug Effectiveness Classification**

In [ ]:
%%capture
!pip install accelerate -U
%%capture
!pip install transformers
%%capture
!pip install evaluate
!pip install datasets
from transformers import Trainer, TrainingArguments, DistilBertForSequenceClassification, \
                         DistilBertTokenizerFast, DataCollatorWithPadding, pipeline
from datasets import Dataset
import evaluate
import pandas as pd
import numpy as np
import os
os.environ["WANDB_DISABLED"] = "true"
from sklearn.pipeline import Pipeline

# Build pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=3000)),
    ('clf', LogisticRegression(max_iter=300))
])

# Train model
pipeline.fit(X_train, y_train)

# Predict
y_pred = pipeline.predict(X_test)

# Evaluation
print("Classification Report:\n")
print(classification_report(y_test, y_pred))



## Import libraries

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Load data
Drug_lib_df = pd.read_csv("drugLibTest_NLP Final Project Dataset.csv")

# Map effectiveness to categories
def map_effectiveness(score):
    if score >= 8:
        return "High"
    elif score >= 5:
        return "Medium"
    else:
        return "Low"

Drug_lib_df['label'] = Drug_lib_df['rating'].apply(map_effectiveness)

# Handle missing values in 'benefitsReview' column before splitting
# Impute missing values with empty strings or drop rows with missing values
Drug_lib_df['benefitsReview'].fillna('', inplace=True)  # Impute with empty strings

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(Drug_lib_df['benefitsReview'], Drug_lib_df['label'], test_size=0.2, random_state=42)

# TF-IDF vectorization
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Logistic Regression
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_tfidf, y_train)
y_pred = clf.predict(X_test_tfidf)

# Classification Report
print("TF-IDF + Logistic Regression Results:")
print(classification_report(y_test, y_pred, target_names=["Low", "Medium", "High"]))

In [ ]:
#1. Imports
import os
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline
)


In [ ]:
#2. Load and Prepare Data

Drug_lib_df = pd.read_csv("drugLibTest_NLP Final Project Dataset.csv")
# Use benefitReview as the text input and rating as the target
df = Drug_lib_df[['benefitsReview', 'rating']].dropna()

# Map ratings to categorical labels
def map_effectiveness(rating):
    if rating >= 8:
        return 2  # High
    elif rating >= 5:
        return 1  # Medium
    else:
        return 0  # Low

df['label'] = df['rating'].apply(map_effectiveness)


In [ ]:
#3. Train/Test Split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['benefitsReview'], df['label'], test_size=0.2, random_state=42
)


In [ ]:
#4. Tokenization
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=128)


In [ ]:
#5. Dataset Formatting
class DrugReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
            'labels': torch.tensor(self.labels[idx])
        }
    def __len__(self):
        return len(self.labels)

train_dataset = DrugReviewDataset(train_encodings, train_labels.tolist())
test_dataset = DrugReviewDataset(test_encodings, test_labels.tolist())


In [ ]:
#6. Training Setup (Disable wandb)
os.environ["WANDB_DISABLED"] = "true"

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=3
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)


In [ ]:
#7. Train the Model
trainer.train()


In [ ]:
#8. Evaluate the Model
results = trainer.evaluate()
print("Evaluation Results:", results)


In [ ]:
#9. Classification Report
pred_output = trainer.predict(test_dataset)
preds = np.argmax(pred_output.predictions, axis=1)

print("Classification Report:")
print(classification_report(test_labels, preds, target_names=["Low", "Medium", "High"]))
